In [1]:
import pandas as pd
import torch
import os
import numpy as np

In [2]:
from datasets import Dataset
from transformers import BertTokenizer, TrainingArguments, Trainer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/preparation/modeldata.csv")

In [4]:
print(df.columns)
print(df.shape)

Index(['verse_id', 'song_id', 'ori_track_name', 'clean_track_name',
       'all_artists', 'primary_artist', 'artist_genres', 'main_genre',
       'explicit', 'section', 'verse', 'language', 'language.1', 'confidence',
       'confidence.1', 'label'],
      dtype='str')
(22878, 16)


In [5]:
# Class balance check
print(df['label'].value_counts())

label
0    11439
1    11439
Name: count, dtype: int64


In [6]:
# Select only the columns we need
df = df[['verse', 'label']]

In [7]:
# Convert to Hugging Face format
dataset = Dataset.from_pandas(df)

In [8]:
# Fixes the [WinError 3] by explicitly setting a local cache directory
cache_dir = "C:/Users/User/Documents/devanasokan_fyp/huggingface_cache"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

In [9]:
# Tell Hugging Face to use this directory
os.environ['HF_HOME'] = cache_dir

# This turns off the annoying symlink warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [10]:
# Initiate tokenizer with the cache path
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased', cache_dir=cache_dir)

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\Documents\devanasokan_fyp\huggingface_cache\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [11]:
def preprocess_function(examples):
    # This creates the 'input_ids' and 'attention_mask' BERT needs
    return tokenizer(examples["verse"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 22878/22878 [00:05<00:00, 4453.94 examples/s]


In [12]:
# 80% Train, 20% Test
full_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

In [13]:
print(full_dataset) 
# If it shows {'train': ..., 'test': ...}, it is already split!

DatasetDict({
    train: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 18302
    })
    test: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4576
    })
})


In [14]:
model_name = "bert-base-uncased" # Or any model from the Hugging Face Hub

# 1. Load the tokenizer (must match the model)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Load the model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2893.24it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

In [15]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [16]:
import evaluate
metric = evaluate.load("accuracy")

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # argmax picks the highest probability (0 or 1)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [18]:
# Set training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Folder where checkpoints are saved
    eval_strategy="epoch",           # Run evaluation after every epoch
    save_strategy="epoch",           # Save model after every epoch
    learning_rate=2e-5,               # Default value; overridden in the final run
    per_device_train_batch_size=16,   # Default value; overridden in the final run
    per_device_eval_batch_size=16,    # Batch size for evaluation
    num_train_epochs=2,               # Default value; overridden in the final run
    weight_decay=0.01,                # Regularization to prevent overfitting
    load_best_model_at_end=True,      # Keeps the best version of the model
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()    # Use Mixed Precision if on GPU for 2x speed
)


In [19]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
)

print(trainer.compute_metrics)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4173.67it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

<function compute_metrics at 0x000001D3B43E7A60>


In [20]:
# Hyperparameter search space definition
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [8, 16]
        ),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    }

# Run the hyperparameter search
best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=hp_space,
    n_trials=6,
    compute_objective=lambda metrics: metrics["eval_accuracy"],
)

print("Best run: ", best_run)
print("Best hyperparameters found: ", best_run.hyperparameters)

[I 2026-06-30 22:36:46,441] A new study created in memory with name: no-name-5457c686-a56c-412b-a227-222085725e8c
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5560.63it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

Epoch,Training Loss,Validation Loss,Accuracy
1,0.335237,0.392449,0.865166
2,0.224458,0.500622,0.883523
3,0.111323,0.563686,0.879808
4,0.043331,0.715480,0.885927


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.28s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

Epoch,Training Loss,Validation Loss,Accuracy
1,0.327037,0.290142,0.872596
2,0.189857,0.377565,0.886364
3,0.121759,0.446422,0.888112
4,0.065504,0.571686,0.889642


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.14s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

...

Epoch,Training Loss,Validation Loss,Accuracy
1,0.319544,0.282329,0.873907
2,0.173317,0.352021,0.884615


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

Epoch,Training Loss,Validation Loss,Accuracy
1,0.362404,0.421913,0.854240


[I 2026-07-01 03:50:53,617] Trial 5 pruned. 


Best run:  BestRun(run_id='1', objective=0.8896416083916084, hyperparameters={'learning_rate': 1.6754008822474014e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 4, 'weight_decay': 0.09465040377562249}, run_summary=None)
Best hyperparameters found:  {'learning_rate': 1.6754008822474014e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 4, 'weight_decay': 0.09465040377562249}


In [21]:
best_hyperparameters = best_run.hyperparameters
final_training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_hyperparameters["learning_rate"],
    per_device_train_batch_size=best_hyperparameters["per_device_train_batch_size"],
    per_device_eval_batch_size=16,
    num_train_epochs=best_hyperparameters["num_train_epochs"],
    weight_decay=best_hyperparameters["weight_decay"],
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

final_trainer = Trainer(
    model=model_init(),
    args=final_training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
)

final_trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9221.82it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

Epoch,Training Loss,Validation Loss,Accuracy
1,0.325008,0.286312,0.876967
2,0.188947,0.359674,0.887893
3,0.119534,0.444375,0.888112
4,0.063894,0.560234,0.887893


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=4576, training_loss=0.17850087400708164, metrics={'train_runtime': 3618.6493, 'train_samples_per_second': 20.231, 'train_steps_per_second': 1.265, 'total_flos': 1.926183414079488e+16, 'train_loss': 0.17850087400708164, 'epoch': 4.0})

In [22]:
history = pd.DataFrame(final_trainer.state.log_history)
print(history)


        loss  grad_norm  learning_rate     epoch  step  eval_loss  \
0   0.380265   6.828809   1.493435e-05  0.437063   500        NaN   
1   0.325008   8.837276   1.310738e-05  0.874126  1000        NaN   
2        NaN        NaN            NaN  1.000000  1144   0.286312   
3   0.227722  10.096125   1.127674e-05  1.311189  1500        NaN   
4   0.188947   8.938389   9.446098e-06  1.748252  2000        NaN   
5        NaN        NaN            NaN  2.000000  2288   0.359674   
6   0.153806   6.638891   7.619120e-06  2.185315  2500        NaN   
7   0.119534  35.319592   5.792142e-06  2.622378  3000        NaN   
8        NaN        NaN            NaN  3.000000  3432   0.444375   
9   0.107373   5.226294   3.961503e-06  3.059441  3500        NaN   
10  0.060625   0.038329   2.130864e-06  3.496503  4000        NaN   
11  0.063894  25.817404   3.002248e-07  3.933566  4500        NaN   
12       NaN        NaN            NaN  4.000000  4576   0.560234   
13       NaN        NaN           

In [23]:
# Save the version currently in the final trainer's brain
final_trainer.save_model("./my_final_model")


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.37s/it]


In [24]:
# TEST MODEL

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Load the model and tokenizer from your local folder
path = "./my_final_model"
model = AutoModelForSequenceClassification.from_pretrained(path)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Ensure you saved the tokenizer there too!

# Create a 'pipeline' (the easiest way to use the model)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2261.35it/s]


In [29]:
# Test it on a new sentence
result = classifier("bla bla bla")
print(result)

[{'label': 'LABEL_0', 'score': 0.9967672824859619}]


In [26]:
import accelerate
print(accelerate.__version__)

1.13.0


In [27]:
import accelerate
import transformers
print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")

Accelerate version: 1.13.0
Transformers version: 5.3.0


In [28]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
